# Lab 04: Data pipeline and quality

## 1. Problem definition

The task is binary classification of annual income above $50,000 using the UCI Adult dataset distributed as OpenML Adult version 2. Each observation has 14 input features: age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week and native-country. The output is one of two income categories, with >50K treated as the positive class. This is an educational study of a historical census sample rather than a proposal to automate decisions about people.

Success is measured with accuracy and positive-class F1. Accuracy reports the fraction of correct predictions, while F1 balances precision and recall for the higher-income class. Because income classes are imbalanced, accuracy alone can obscure poor performance on the positive class. We compare a fixed Logistic Regression baseline using the original inputs with the same classifier augmented by two predefined features. The test set is used for this prescribed comparison, not for hyperparameter tuning or selecting among additional features.

The main constraints are reproducibility and avoiding information leakage in fitted preprocessing. We use a fixed seed and stratified 70/30 split, preserve the ingested data, fit every imputation value, clipping bound, scaling statistic and encoding vocabulary on training observations only, and retain every test observation. The pipeline reports missingness, invalid types or categories, range violations, exact duplicates and conflicting income labels. It saves the fitted transformer, configuration and a SHA256 hash of a precisely defined processed table. A fresh run requires network access for OpenML; subsequent runs use its local cache. Exact byte reproducibility is tied to the recorded library versions and source snapshot.


Run all cells from top to bottom. The small transforms.py module defines the serializable preprocessing class used by joblib.

One-command run from the repository root (same course Python environment):
```powershell
.\lab01\.venv\Scripts\python.exe -c "import json; n=json.load(open('lab04/data_pipeline.ipynb',encoding='utf-8')); [exec(compile(''.join(c['source']),'<notebook>','exec'),globals()) for c in n['cells'] if c['cell_type']=='code']"
```

In [1]:
import hashlib
import json
import platform
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sys
sys.path.insert(0, str(Path.cwd() if Path('transforms.py').exists() else Path.cwd() / 'lab04'))
from transforms import NUMERIC, CATEGORICAL, CleanFeatures

ROOT = Path.cwd() if Path('transforms.py').exists() else Path.cwd() / 'lab04'
OUT = ROOT / 'results'
SEED = 42
RANGES = {'age': [17, 90], 'hours-per-week': [1, 99], 'capital-gain': [0, 99999], 'capital-loss': [0, 4356]}
# Fixed UCI dictionary, independent of observed/test values; '?' is permitted missingness.
DOMAINS = {
    'workclass': 'Private,Self-emp-not-inc,Self-emp-inc,Federal-gov,Local-gov,State-gov,Without-pay,Never-worked',
    'education': 'Bachelors,Some-college,11th,HS-grad,Prof-school,Assoc-acdm,Assoc-voc,9th,7th-8th,12th,Masters,1st-4th,10th,Doctorate,5th-6th,Preschool',
    'marital-status': 'Married-civ-spouse,Divorced,Never-married,Separated,Widowed,Married-spouse-absent,Married-AF-spouse',
    'occupation': 'Tech-support,Craft-repair,Other-service,Sales,Exec-managerial,Prof-specialty,Handlers-cleaners,Machine-op-inspct,Adm-clerical,Farming-fishing,Transport-moving,Priv-house-serv,Protective-serv,Armed-Forces',
    'relationship': 'Wife,Own-child,Husband,Not-in-family,Other-relative,Unmarried',
    'race': 'White,Asian-Pac-Islander,Amer-Indian-Eskimo,Other,Black',
    'sex': 'Female,Male',
    'native-country': 'United-States,Cambodia,England,Puerto-Rico,Canada,Germany,Outlying-US(Guam-USVI-etc),India,Japan,Greece,South,China,Cuba,Iran,Honduras,Philippines,Italy,Poland,Jamaica,Vietnam,Mexico,Portugal,Ireland,France,Dominican-Republic,Laos,Ecuador,Taiwan,Haiti,Columbia,Hungary,Guatemala,Nicaragua,Scotland,Thailand,Yugoslavia,El-Salvador,Trinadad&Tobago,Peru,Hong,Holand-Netherlands',
    'income': '<=50K,>50K',
}


def save_json(name, obj):
    (OUT / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')


def validate(raw):
    issues = []
    missing = {}
    for c in raw:
        s = raw[c]
        missing[c] = int((s.isna() | s.eq('?')).sum())
        if c in NUMERIC:
            v = pd.to_numeric(s, errors='coerce')
            bad = s.notna() & (v.isna() | (v % 1 != 0))
        else:
            bad = s.notna() & ~s.isin(DOMAINS[c].split(',') + ['?'])
        for idx in raw.index[bad]:
            issues.append({'row_id': int(idx), 'column': c, 'check': 'type_or_category', 'value': str(s.loc[idx])})
        if c in RANGES:
            lo, hi = RANGES[c]
            for idx in raw.index[s.notna() & ~pd.to_numeric(s, errors='coerce').between(lo, hi)]:
                issues.append({'row_id': int(idx), 'column': c, 'check': 'range', 'value': str(s.loc[idx])})
    return issues, missing

## 2. Ingestion
Read Adult version 2 without modifying the source. Expected shape: 48,842 rows and 15 columns.

In [2]:
pd.set_option('future.no_silent_downcasting', True)
OUT.mkdir(exist_ok=True)
ds = fetch_openml(name='adult', version=2, as_frame=True, data_home=str(ROOT.parent / 'data' / 'openml'))
# Ingestion is read-only; rename only on a separate working copy.
original = ds.frame
before = hashlib.sha256(original.to_csv(index=False).encode()).hexdigest()
assert original.shape == (48842, 15)
raw = original.rename(columns={ds.target.name: 'income'}).copy(deep=True)

print('Raw shape:', original.shape)

Raw shape: (48842, 15)


## 3. Validation
Check integer values, categories, specified ranges and missing values. The fixed allowed domains follow the UCI dictionary; ? is an allowed missing marker. No rows are removed by validation.

In [3]:
issues, missing = validate(raw)
pd.DataFrame(issues, columns=['row_id', 'column', 'check', 'value']).to_csv(OUT / 'violations.csv', index=False)
if raw.income.isna().any() or not raw.income.isin(['<=50K', '>50K']).all():
    raise ValueError('Invalid labels: report rather than silently removing rows')

print('Violations:', len(issues))
print('Missing values:', missing)

Violations: 0
Missing values: {'age': 0, 'workclass': 2799, 'fnlwgt': 0, 'education': 0, 'education-num': 0, 'marital-status': 0, 'occupation': 2809, 'relationship': 0, 'race': 0, 'sex': 0, 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 0, 'native-country': 857, 'income': 0}


## 4. Cleaning and transformation
Split 70/30 with stratification, remove exact duplicates from train only, and define preprocessing: numeric median and categorical mode imputation, 1st/99th percentile clipping, scaling and one-hot encoding. Fitted preprocessing statistics use train only.


In [4]:
train, test = train_test_split(raw, test_size=.30, random_state=SEED, stratify=raw.income)
test_snapshot = test.copy(deep=True)
n_before = len(train)
train = train.drop_duplicates()  # complete rows, including income
feature_cols = list(ds.data.columns)

print('Train before/after dedup:', n_before, len(train))
print('Test rows:', len(test))

def make_preprocessor(derived=False):
    numbers = NUMERIC + (['capital_net', 'hours_per_age'] if derived else [])
    return Pipeline([
        ('clean', CleanFeatures(derived)),
        ('encode', ColumnTransformer([
            ('numeric', StandardScaler(), numbers),
            ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=True), CATEGORICAL),
        ])),
    ])


Train before/after dedup: 34189 34165
Test rows: 14653


## 5. Derived features and model comparison
Compare LogisticRegression with and without capital_net = capital-gain - capital-loss and hours_per_age = hours-per-week / (age - 16). Features use the imputed and clipped inputs. Fit each preprocessor on train only and save the fitted engineered preprocessor.


In [5]:
metrics = []
for derived in [False, True]:
    name = 'engineered' if derived else 'baseline'
    processor = make_preprocessor(derived)
    model = Pipeline([('preprocessor', processor), ('classifier', LogisticRegression(solver='lbfgs', max_iter=3000, random_state=SEED))])
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(train[feature_cols], train.income.eq('>50K').astype(int))
    if any(issubclass(w.category, ConvergenceWarning) for w in recorded):
        raise RuntimeError('LogisticRegression did not converge')
    y = test.income.eq('>50K').astype(int)
    pred = model.predict(test[feature_cols])
    metrics.append({'model': name, 'accuracy': accuracy_score(y, pred), 'f1': f1_score(y, pred), 'iterations': int(model[-1].n_iter_[0])})
    if derived:
        joblib.dump(processor, OUT / 'preprocessor.joblib')
pd.DataFrame(metrics).to_csv(OUT / 'metrics.csv', index=False)
pd.testing.assert_frame_equal(test, test_snapshot)
assert before == hashlib.sha256(original.to_csv(index=False).encode()).hexdigest()

print(pd.DataFrame(metrics).to_string(index=False))

     model  accuracy       f1  iterations
  baseline  0.853955 0.660102          61
engineered  0.857777 0.669626          75


## 6. Label quality
Find identical feature vectors with conflicting income labels. Export a seeded random sample of 50 rows for manual checking against the [UCI dictionary](https://archive.ics.uci.edu/dataset/2/adult). Saved statuses and notes are loaded from results/review_50.csv and retained on reruns when the sampled records are unchanged. Do not infer whether the income label is correct from demographic features.

In [6]:
groups = raw.groupby(feature_cols, observed=True, dropna=False).income.agg(['nunique', 'size'])
conflicts = groups[groups['nunique'] > 1].reset_index()
conflicts.to_csv(OUT / 'label_conflicts.csv', index=False)
sample = raw.sample(n=50, random_state=SEED).copy()
sample.insert(0, 'row_id', sample.index)
issue_ids = {x['row_id'] for x in issues}
sample['automated_schema_range_check'] = ['flagged' if i in issue_ids else 'pass' for i in sample.index]
sample['human_review_status'] = 'pending'
sample['human_review_notes'] = ''
review_path = OUT / 'review_50.csv'
# Retain existing review entries only when the sampled source row is unchanged.
if review_path.exists():
    saved = pd.read_csv(review_path).set_index('row_id')
    for idx in sample.index:
        if idx not in saved.index:
            continue
        current = sample.loc[idx, list(raw.columns)].astype(object).fillna('').astype(str)
        previous = saved.loc[idx, list(raw.columns)].astype(object).fillna('').astype(str)
        if current.equals(previous):
            for column in ['human_review_status', 'human_review_notes']:
                value = saved.loc[idx, column]
                if pd.notna(value):
                    sample.loc[idx, column] = value
sample.to_csv(review_path, index=False)

print('Conflicting feature groups:', len(conflicts))
print(conflicts.to_string(index=False))
print(sample.to_string(index=False))

Conflicting feature groups: 5
 age workclass  fnlwgt education  education-num     marital-status       occupation  relationship  race    sex  capital-gain  capital-loss  hours-per-week native-country  nunique  size
  34   Private  169583   Masters             14      Never-married  Exec-managerial Not-in-family White Female             0             0              50  United-States        2     2
  39   Private  138192 Bachelors             13 Married-civ-spouse     Craft-repair       Husband White   Male             0             0              40  United-States        2     2
  41   Private  242804   HS-grad              9 Married-civ-spouse Transport-moving       Husband White   Male             0             0              40  United-States        2     2
  46   Private  224559   HS-grad              9 Married-civ-spouse Transport-moving       Husband Black   Male             0             0              40  United-States        2     2
  46   Private  235646   HS-grad             

## Review results
The review table contains 50 rows marked pass. No impossible values were found. Three rows contain allowed missing values: 21063 and 18957 lack workclass and occupation; 20139 lacks native-country. The 4 and 84 hours/week values are unusual but within the allowed range. Notes are saved in results/review_50.csv.

Income labels cannot be verified from demographic features alone. The duplicate-conflict audit found five feature groups with conflicting labels (10 records).

## 7. Validation report, SHA256 and configuration
Hash the exact UTF-8 processed.csv bytes (cleaned and engineered, before encoding; includes row ID, split and label). Save the hash with YAML configuration. See data_card.md for source, license, size, biases and privacy.

In [7]:
chunks = []
for split, frame in [('train', train), ('test', test)]:
    processed = processor.named_steps['clean'].transform(frame[feature_cols])
    processed.insert(0, 'row_id', frame.index)
    processed.insert(1, 'split', split)
    processed['income'] = frame.income
    chunks.append(processed)
# Hash a canonical cleaned, clipped, engineered table before sparse encoding.
path = OUT / 'processed.csv'
pd.concat(chunks).to_csv(path, index=False, float_format='%.17g', lineterminator='\n')
digest = hashlib.sha256(path.read_bytes()).hexdigest()

audit = {'raw_shape': list(raw.shape), 'type_or_range_violations': len(issues), 'missing_counts': missing,
         'exact_duplicate_excess_raw': int(raw.duplicated().sum()), 'train_before_dedup': n_before,
         'train_after_dedup': len(train), 'removed_train_duplicates': n_before-len(train), 'test_rows': len(test),
         'conflicting_feature_groups': len(conflicts), 'rows_in_conflicting_groups': int(conflicts['size'].sum()),
 'manual_review': {'rows': len(sample), 'status_counts': sample['human_review_status'].value_counts().to_dict(), 'rows_with_missing_values': int(sample[list(raw.columns)].isna().any(axis=1).sum())},
         'raw_unchanged': True, 'test_rows_unchanged': True}
save_json('validation_report.json', audit)
config = {'dataset': {'name': 'adult', 'version': 2, 'openml_id': int(ds.details['id'])},
          'seed': SEED, 'test_fraction': .3, 'stratified': True, 'deduplication': 'train complete rows only',
          'schema': {**{c: 'integer' for c in NUMERIC}, **{c: 'category' for c in CATEGORICAL + ['income']}},
          'allowed_categories': {c: v.split(',') + ['?'] for c, v in DOMAINS.items()}, 'ranges': RANGES,
          'imputation': {'numeric': 'train median', 'categorical': 'train mode'}, 'clip_quantiles': [.01, .99],
          'feature_order': 'impute, clip original numeric, derive, scale/one-hot',
          'derived': {'capital_net': 'capital-gain - capital-loss', 'hours_per_age': 'hours-per-week / (age - 16)'},
          'logistic_regression': {'solver': 'lbfgs', 'max_iter': 3000, 'C': 1.0, 'threshold': .5},
          'processed_file': 'processed.csv', 'processed_representation': 'cleaned/clipped/engineered, before encoding; train then test; includes row_id, split, income',
          'processed_sha256': digest, 'raw_frame_sha256': before,
          'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'scipy': scipy.__version__, 'joblib': joblib.__version__, 'pyyaml': yaml.__version__}}
(OUT / 'config.yaml').write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
(OUT / 'processed.sha256').write_text(digest + '  processed.csv\n', encoding='ascii')
print(json.dumps({'audit': audit, 'metrics': metrics, 'sha256': digest}, indent=2))

{
  "audit": {
    "raw_shape": [
      48842,
      15
    ],
    "type_or_range_violations": 0,
    "missing_counts": {
      "age": 0,
      "workclass": 2799,
      "fnlwgt": 0,
      "education": 0,
      "education-num": 0,
      "marital-status": 0,
      "occupation": 2809,
      "relationship": 0,
      "race": 0,
      "sex": 0,
      "capital-gain": 0,
      "capital-loss": 0,
      "hours-per-week": 0,
      "native-country": 857,
      "income": 0
    },
    "exact_duplicate_excess_raw": 52,
    "train_before_dedup": 34189,
    "train_after_dedup": 34165,
    "removed_train_duplicates": 24,
    "test_rows": 14653,
    "conflicting_feature_groups": 5,
    "rows_in_conflicting_groups": 10,
    "manual_review": {
      "rows": 50,
      "status_counts": {
        "pass": 50
      },
      "rows_with_missing_values": 3
    },
    "raw_unchanged": true,
    "test_rows_unchanged": true
  },
  "metrics": [
    {
      "model": "baseline",
      "accuracy": 0.8539548215382515,
   